# 🚀 AutoGluon v2 — Ultimate Demand Forecasting
ผสมทุก insight จากทุกไฟล์: Leakage-Safe Lags, Momentum, Stockout, Events, Annual Lags, B1G1, Interaction Features
**Run on GPU (T4x2). ไม่ต้องติดตั้งอะไรล่วงหน้า**

In [1]:
import subprocess, sys
try:
    import autogluon.tabular
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "autogluon", "-q"])
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.9.1 which is incompatible.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.1 which is incompatible.


In [2]:
# ─── Paths ──────────────────────────────────────────────────────────
DATA_DIR = Path("/kaggle/input/competitions/super-ai-engineer-season-6-coffee-chain-hackathon")
TR, TE = DATA_DIR / "train", DATA_DIR / "test"
print("Data:", DATA_DIR)

def rc(p): return pd.read_csv(p).rename(columns=str.strip)

txn       = rc(TR/"TRANSACTION.csv")
order     = rc(TR/"ORDER.csv")
inventory = rc(TR/"INVENTORY.csv")
product_tr = rc(TR/"PRODUCT.csv")
product_te = rc(TE/"PRODUCT.csv")
store_tr   = rc(TR/"STORE.csv")
store_te   = rc(TE/"STORE.csv")
date_tr    = rc(TR/"DATE_DIM.csv")
date_te    = rc(TE/"DATE_DIM.csv")
promo_tr   = rc(TR/"PROMOTION.csv")
promo_te   = rc(TE/"PROMOTION.csv")
event_tr   = rc(TR/"LOCAL_EVENT.csv")
event_te   = rc(TE/"LOCAL_EVENT.csv")
sample     = rc(DATA_DIR/"sample_submission_with_id.csv")

for df, cols in [(order,["date"]), (inventory,["date"]), (promo_tr,["start_date","end_date"]),
                 (promo_te,["start_date","end_date"]), (event_tr,["date"]), (event_te,["date"])]:
    for c in cols:
        if c in df.columns: df[c] = pd.to_datetime(df[c])

Data: /kaggle/input/competitions/super-ai-engineer-season-6-coffee-chain-hackathon


In [3]:
# ─── Target ─────────────────────────────────────────────────────────
product = pd.concat([product_tr, product_te]).drop_duplicates("product_id")
target = (txn[["order_id","product_id","units_sold"]]
          .merge(order[["order_id","store_id","date"]], on="order_id")
          .merge(product[["product_id","category"]], on="product_id")
          .groupby(["store_id","category","date"], as_index=False)["units_sold"].sum())
target["date"] = pd.to_datetime(target["date"])

STORES = sorted(target["store_id"].unique())
CATS   = sorted(target["category"].unique())
DATES  = pd.date_range("2023-01-01", "2024-10-31", freq="D")

grid = pd.MultiIndex.from_product([STORES, CATS, DATES],
       names=["store_id","category","date"]).to_frame(index=False)
full = grid.merge(target, on=["store_id","category","date"], how="left").fillna({"units_sold":0})
print(f"Grid: {full.shape}")

Grid: (93800, 4)


In [4]:
# ─── Store Features ─────────────────────────────────────────────────
store = pd.concat([store_tr, store_te]).drop_duplicates("store_id")
store["open_h"]  = pd.to_numeric(store["open_time"].str.split(":").str[0], errors="coerce").fillna(7)
store["close_h"] = pd.to_numeric(store["close_time"].str.split(":").str[0], errors="coerce").fillna(21)
store["op_hours"]      = store["close_h"] - store["open_h"]
store["cap_per_staff"] = store["seating_capacity"] / store["staff_count"].replace(0, np.nan)
store["has_drive"]     = store["has_drive_through"].astype(str).str.lower().isin(["true","1","y"]).astype(int)
store["store_age"]     = (pd.Timestamp("2024-11-01") - pd.to_datetime(store["opened_date"])).dt.days
store["is_mall"]       = (store["neighborhood_type"] == "mall").astype(int)
store["is_tourist"]    = (store["neighborhood_type"] == "tourist").astype(int)
store["is_university"] = (store["neighborhood_type"] == "university").astype(int)
store["is_hospital"]   = (store["neighborhood_type"] == "hospital").astype(int)
nbr = pd.get_dummies(store["neighborhood_type"], prefix="nbr").astype(int)
for c in ["nbr_airport","nbr_business","nbr_hospital","nbr_mall","nbr_residential","nbr_tourist","nbr_university"]:
    if c not in nbr: nbr[c] = 0
store_feats = pd.concat([store[["store_id","seating_capacity","staff_count","op_hours",
                                  "cap_per_staff","has_drive","store_age",
                                  "is_mall","is_tourist","is_university","is_hospital"]], nbr], axis=1)
full = full.merge(store_feats, on="store_id", how="left")

In [5]:
# ─── Calendar Features ──────────────────────────────────────────────
date_all = pd.concat([date_tr, date_te]).drop_duplicates("date")
date_all["date"] = pd.to_datetime(date_all["date"])
date_all["dow"]  = date_all["date"].dt.dayofweek
date_all["doy"]  = date_all["date"].dt.dayofyear
date_all["dom"]  = date_all["date"].dt.day
date_all["woy"]  = date_all["date"].dt.isocalendar().week.astype(int)
date_all["month"]    = date_all["date"].dt.month
date_all["quarter"]  = date_all["date"].dt.quarter
date_all["sin_dow"]  = np.sin(2*np.pi*date_all["dow"]/7)
date_all["cos_dow"]  = np.cos(2*np.pi*date_all["dow"]/7)
date_all["sin_doy"]  = np.sin(2*np.pi*date_all["doy"]/365.25)
date_all["cos_doy"]  = np.cos(2*np.pi*date_all["doy"]/365.25)
date_all["is_dec"]   = (date_all["month"] == 12).astype(int)
date_all["is_nov"]   = (date_all["month"] == 11).astype(int)
date_all["is_newyear"] = date_all["date"].dt.strftime("%m-%d").isin(["12-31","01-01"]).astype(int)
for c in ["is_weekend","is_holiday","is_payday","is_school_break","is_rainy_season"]:
    if c in date_all.columns:
        date_all[c] = date_all[c].astype(str).str.lower().isin(["true","1","y"]).astype(int)
    else:
        date_all[c] = 0

cal_cols = ["date","dow","doy","dom","woy","month","quarter","sin_dow","cos_dow",
            "sin_doy","cos_doy","is_dec","is_nov","is_newyear",
            "is_weekend","is_holiday","is_payday","is_school_break","is_rainy_season"]
full = full.merge(date_all[cal_cols], on="date", how="left")

In [6]:
# ─── Promo Features ─────────────────────────────────────────────────
promo = pd.concat([promo_tr, promo_te]).drop_duplicates()
promo = promo.merge(product[["product_id","category"]], on="product_id", how="left")
promo["start_date"] = pd.to_datetime(promo["start_date"])
promo["end_date"]   = pd.to_datetime(promo["end_date"])

# detect B1G1
camp_col = next((c for c in ["campaign_name","campaign_type","promo_type"] if c in promo.columns), None)
promo["is_b1g1"] = 0
if camp_col:
    promo["is_b1g1"] = promo[camp_col].fillna("").str.lower().str.contains("1แถม1|buy1get1|b1g1").astype(int)

p_rows = []
for _, r in promo.iterrows():
    if pd.isna(r["start_date"]) or pd.isna(r.get("category")): continue
    s = max(r["start_date"], pd.Timestamp("2023-01-01"))
    e = min(r["end_date"],   pd.Timestamp("2024-10-31"))
    if e < s: continue
    sid = int(r["store_id"]) if not pd.isna(r.get("store_id")) else None
    for d in pd.date_range(s, e):
        row = {"store_id": sid or -1, "category": r["category"], "date": d,
               "promo_active": 1, "discount_pct": float(r.get("discount_pct",0) or 0),
               "is_b1g1": int(r["is_b1g1"])}
        p_rows.append(row)

if p_rows:
    pf = pd.DataFrame(p_rows)
    pf_grp = pf.groupby(["store_id","category","date"], as_index=False).agg(
        promo_active=("promo_active","max"), promo_count=("promo_active","sum"),
        max_discount=("discount_pct","max"), mean_discount=("discount_pct","mean"),
        is_b1g1=("is_b1g1","max"))
    # global promos (store_id=-1) → broadcast
    global_p = pf_grp[pf_grp["store_id"]==-1].drop(columns="store_id")
    store_p  = pf_grp[pf_grp["store_id"]!=-1]
    full = full.merge(store_p, on=["store_id","category","date"], how="left")
    full = full.merge(global_p, on=["category","date"], how="left", suffixes=("","_g"))
    for c in ["promo_active","promo_count","max_discount","mean_discount","is_b1g1"]:
        gc = c+"_g"
        if gc in full.columns:
            full[c] = full[c].combine_first(full[gc])
            full.drop(columns=gc, inplace=True)
full[["promo_active","promo_count","max_discount","mean_discount","is_b1g1"]] =     full[["promo_active","promo_count","max_discount","mean_discount","is_b1g1"]].fillna(0)

In [7]:
# ─── Event Features ─────────────────────────────────────────────────
event = pd.concat([event_tr, event_te]).drop_duplicates()
event["date"] = pd.to_datetime(event["date"])
ev_type = pd.get_dummies(event["event_type"], prefix="evt").astype(int)
event = pd.concat([event[["store_id","date"]], ev_type], axis=1)
ev_agg = event.groupby(["store_id","date"], as_index=False).agg(
    n_events=("store_id","count"), **{c: (c,"max") for c in ev_type.columns})
ev_agg["has_event"] = 1
full = full.merge(ev_agg, on=["store_id","date"], how="left")
full["n_events"]  = full["n_events"].fillna(0)
full["has_event"] = full["has_event"].fillna(0)
for c in [c for c in full.columns if c.startswith("evt_")]:
    full[c] = full[c].fillna(0)
print("Event cols:", [c for c in full.columns if c.startswith("evt_")])

Event cols: ['evt_book_fair', 'evt_concert', 'evt_convention', 'evt_cultural', 'evt_food_festival', 'evt_market', 'evt_music_festival', 'evt_sports']


In [8]:
# ─── Stockout Features ──────────────────────────────────────────────
inv = inventory.merge(product[["product_id","category"]], on="product_id", how="left")
inv["date"] = pd.to_datetime(inv["date"])
stockout = inv.groupby(["store_id","category","date"], as_index=False).agg(
    stockout_rate=("is_stockout","mean"),
    avg_closing_stock=("closing_stock","mean"))
full = full.merge(stockout, on=["store_id","category","date"], how="left").fillna(
    {"stockout_rate":0, "avg_closing_stock":0})

In [9]:
# ─── Sorting for lag/rolling ─────────────────────────────────────────
full = full.sort_values(["store_id","category","date"]).reset_index(drop=True)
g = full.groupby(["store_id","category"])

# Category OHE
cat_d = pd.get_dummies(full["category"], prefix="cat").astype(int)
full = pd.concat([full, cat_d], axis=1)

# Interaction features
full["b1g1_x_coffee"]   = full["is_b1g1"] * full.get("cat_Coffee", 0)
full["b1g1_x_tea"]      = full["is_b1g1"] * full.get("cat_Tea", 0)
full["rainy_x_coffee"]  = full["is_rainy_season"] * full.get("cat_Coffee", 0)
full["rainy_x_juice"]   = full["is_rainy_season"] * full.get("cat_Juice & Smoothie", 0)
full["holiday_x_coffee"]= full["is_holiday"] * full.get("cat_Coffee", 0)
full["event_x_bakery"]  = full["has_event"] * full.get("cat_Bakery", 0)
full["mall_x_weekend"]  = full["is_mall"] * full["is_weekend"]
full["tourist_x_holiday"]= full["is_tourist"] * full["is_holiday"]
full["cap_x_weekend"]   = full["seating_capacity"] * full["is_weekend"]
full["payday_x_coffee"] = full["is_payday"] * full.get("cat_Coffee", 0)

# Store-level total (cross-category context)
store_day = full.groupby(["store_id","date"])["units_sold"].sum().reset_index().rename(columns={"units_sold":"store_total"})
store_day["store_total_lag1"]  = store_day.groupby("store_id")["store_total"].shift(1)
store_day["store_total_lag7"]  = store_day.groupby("store_id")["store_total"].shift(7)
store_day["store_roll_mean_7"] = store_day.groupby("store_id")["store_total"].transform(lambda x: x.shift(1).rolling(7,min_periods=1).mean())
full = full.merge(store_day[["store_id","date","store_total_lag1","store_total_lag7","store_roll_mean_7"]],
                  on=["store_id","date"], how="left")

In [10]:
# ─── Product-level features ──────────────────────────────────────────
agg_dict = {
    "product_count": ("product_id", "nunique"),
    "avg_base_price": ("base_price", "mean"),
    "max_base_price": ("base_price", "max"),
}
if "is_seasonal" in product.columns:
    agg_dict["seasonal_rate"] = ("is_seasonal", "mean")
prod_feats = product.groupby("category").agg(**agg_dict).reset_index()
full = full.merge(prod_feats, on="category", how="left")

In [11]:
# ─── Stockout lag features (horizon-specific, computed later) ─────────
# Stored separately for leakage-safe merge in panel builder
print("Base features ready. Full shape:", full.shape)
base_feature_cols = [c for c in full.columns if c not in ["store_id","category","date","units_sold"]]
print(f"Base feature count: {len(base_feature_cols)}")

Base features ready. Full shape: (93800, 84)
Base feature count: 80


## Build Horizon-Specific Panels (Leakage-Safe)

In [12]:
HORIZON_MAP = {"1d":1, "7d":7, "1m":30}

def make_horizon_panel(full, horizon_str, stores, cats, dates):
    H = HORIZON_MAP[horizon_str]
    df = full.copy().sort_values(["store_id","category","date"])

    # ── Strict leakage-safe lags ─────────────────────────────────────
    for k in [0, 1, 2, 7, 14, 21, 28, 35, 56, 84]:
        lag = H + k
        col = f"lag_{lag}"
        df[col] = df.groupby(["store_id","category"])["units_sold"].shift(lag)

    # Annual lags (same week last year → Nov-Dec 2024 maps to Nov-Dec 2023)
    for ann in [364, 365, 366]:
        df[f"lag_{ann}"] = df.groupby(["store_id","category"])["units_sold"].shift(ann)

    # ── Momentum (from friend's 43-feature set) ──────────────────────
    lag_h   = df.groupby(["store_id","category"])["units_sold"].shift(H)
    lag_h7  = df.groupby(["store_id","category"])["units_sold"].shift(H+7)
    lag_h14 = df.groupby(["store_id","category"])["units_sold"].shift(H+14)
    df["lag_ratio_7_14"]  = lag_h7  / (lag_h14  + 1e-5)
    df["lag_diff_7_14"]   = lag_h7  - lag_h14
    df["lag_pct_7_14"]    = (lag_h7 - lag_h14) / (lag_h14 + 1e-5)
    df["lag_diff_h_h7"]   = lag_h   - lag_h7
    df["lag_ratio_h_h7"]  = lag_h   / (lag_h7  + 1e-5)

    # ── Rolling stats (shifted >= H, leakage-safe) ───────────────────
    for w in [7, 14, 28, 56]:
        df[f"roll_mean_{w}"] = df.groupby(["store_id","category"])["units_sold"].transform(
            lambda x, _H=H, _w=w: x.shift(_H).rolling(_w, min_periods=1).mean())
        df[f"roll_std_{w}"]  = df.groupby(["store_id","category"])["units_sold"].transform(
            lambda x, _H=H, _w=w: x.shift(_H).rolling(_w, min_periods=1).std())
        df[f"roll_max_{w}"]  = df.groupby(["store_id","category"])["units_sold"].transform(
            lambda x, _H=H, _w=w: x.shift(_H).rolling(_w, min_periods=1).max())
        df[f"roll_min_{w}"]  = df.groupby(["store_id","category"])["units_sold"].transform(
            lambda x, _H=H, _w=w: x.shift(_H).rolling(_w, min_periods=1).min())
        df[f"roll_nonzero_{w}"] = df.groupby(["store_id","category"])["units_sold"].transform(
            lambda x, _H=H, _w=w: (x.shift(_H) > 0).rolling(_w, min_periods=1).sum())

    # Trend feature (from catboost_summary)
    df["trend_7_vs_28"] = df["roll_mean_7"] - df["roll_mean_28"]
    df["cv_28"]         = df["roll_std_28"] / (df["roll_mean_28"] + 1e-5)
    df["momentum_7_28"] = df["roll_mean_7"] / (df["roll_mean_28"] + 1e-5)
    df["momentum_14_56"]= df["roll_mean_14"] / (df["roll_mean_56"] + 1e-5)

    # ── Stockout lags ────────────────────────────────────────────────
    df["stockout_lag0"]     = df.groupby(["store_id","category"])["stockout_rate"].shift(H)
    df["stockout_lag7"]     = df.groupby(["store_id","category"])["stockout_rate"].shift(H+7)
    df["stockout_roll_28"]  = df.groupby(["store_id","category"])["stockout_rate"].transform(
        lambda x: x.shift(H).rolling(28, min_periods=1).mean())

    # ── Sample weight (stockout rows downweighted) ───────────────────
    df["sample_weight"] = 1.0 - 0.7 * df["stockout_lag0"].fillna(0).clip(0,1)

    df["horizon"] = horizon_str
    df = df.fillna(0)
    return df

panels = {}
for h_str in ["1d","7d","1m"]:
    panels[h_str] = make_horizon_panel(full, h_str, STORES, CATS, DATES)
    ncols = len([c for c in panels[h_str].columns if c not in ["store_id","category","date","units_sold","sample_weight","horizon"]])
    print(f"Horizon {h_str}: {ncols} features")

Horizon 1d: 125 features
Horizon 7d: 125 features
Horizon 1m: 125 features


## AutoGluon Training — Time-Machine Split (Jan-Oct 2023 → Nov-Dec 2023)

In [ ]:
# AutoGluon requires sample_weight as a COLUMN NAME in the DataFrame (not an array)
EXCLUDE = {"store_id","category","date","units_sold","sample_weight","horizon"}
MODELS  = {}
VAL_RESULTS = {}

for h_str in ["1d","7d","1m"]:
    print(f"\n{'='*50}\nTraining AutoGluon — Horizon: {h_str}\n{'='*50}")
    df = panels[h_str].copy()

    TRAIN   = df[df["date"] < "2023-11-01"].copy()
    VALID   = df[df["date"].between("2023-11-01","2023-12-31")].copy()
    RETRAIN = df[df["date"] <= "2024-10-31"].copy()

    feat_cols = [c for c in df.columns if c not in EXCLUDE]
    label_col = "units_sold"
    train_cols = feat_cols + [label_col]  # sample_weight not supported in this AG version

    print(f"  Features: {len(feat_cols)}")
    print(f"  Train rows: {len(TRAIN):,}  |  Valid rows: {len(VALID):,}")

    # ── Validation predictor (Time-Machine Split) ────────────────────
    predictor = TabularPredictor(
        label=label_col,
        problem_type="regression",
        eval_metric="mean_absolute_error",
        path=f"ag_v2_{h_str}",
        verbosity=2,
    ).fit(
        TRAIN[train_cols],
        presets="best_quality",
        time_limit=3600,
        hyperparameters={
            "GBM": [
                {"objective": "tweedie", "tweedie_variance_power": 1.1, "extra_trees": False},
                {"objective": "tweedie", "tweedie_variance_power": 1.5, "extra_trees": True},
            ],
            "CAT": [
                {"loss_function": "MAE",  "depth": 8},
                {"loss_function": "RMSE", "depth": 6},
            ],
            "XGB": [{"objective": "reg:tweedie", "tweedie_variance_power": 1.2}],
            "NN_TORCH": {},
        },
    )

    val_pred = predictor.predict(VALID[feat_cols]).clip(lower=0)
    val_mae  = (val_pred - VALID[label_col]).abs().mean()
    print(f"  [Val MAE {h_str}]: {val_mae:.4f}")
    VAL_RESULTS[h_str] = val_mae

    # Per-model leaderboard (top 5)
    lb = predictor.leaderboard(VALID[feat_cols + [label_col]], silent=True)
    print(lb[["model","score_test","score_val"]].head(5).to_string(index=False))
    lb.to_csv(f"leaderboard_{h_str}.csv", index=False)

    # ── Retrain on full Jan2023–Oct2024 ─────────────────────────────
    print("  Retraining on full data...")
    predictor_full = TabularPredictor(
        label=label_col,
        problem_type="regression",
        eval_metric="mean_absolute_error",
        path=f"ag_v2_{h_str}_full",
        verbosity=0,
    ).fit(
        RETRAIN[train_cols],
        presets="best_quality",
        time_limit=1800,
    )
    MODELS[h_str] = (predictor_full, feat_cols)
    print(f"  Retrain complete")

print("\n=== Validation Summary ===")
for h, v in VAL_RESULTS.items():
    print(f"  {h}: MAE = {v:.4f}")


Training AutoGluon — Horizon: 1d
  Features: 125
  Train rows: 42,560  |  Valid rows: 8,540


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.13 GB, Allocated: 0.00 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       28.48 GB / 31.35 GB (90.8%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable

  [Val MAE 1d]: 10.7288
              model  score_test  score_val
  CatBoost_2_BAG_L1  -10.174782  -7.889476
    CatBoost_BAG_L1  -10.280720  -8.020118
WeightedEnsemble_L2  -10.728780  -7.348522
  LightGBM_2_BAG_L1  -10.760264  -7.369593
     XGBoost_BAG_L1  -10.997277  -7.764770
  Retraining on full data...
  Retrain complete

Training AutoGluon — Horizon: 7d


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.13 GB, Allocated: 0.00 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       26.23 GB / 31.35 GB (83.7%)
Disk Space Avail:   18.70 GB / 19.52 GB (95.8%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable

  Features: 125
  Train rows: 42,560  |  Valid rows: 8,540


...

  [Val MAE 7d]: 10.4209
            model  score_test  score_val
CatBoost_2_BAG_L1  -10.002155  -8.060821
LightGBM_2_BAG_L2  -10.246766  -7.711911
  CatBoost_BAG_L2  -10.284153  -7.681513
   XGBoost_BAG_L1  -10.308348  -7.955040
CatBoost_2_BAG_L2  -10.315323  -7.666378
  Retraining on full data...


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.13 GB, Allocated: 0.00 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       25.96 GB / 31.35 GB (82.8%)
Disk Space Avail:   17.94 GB / 19.52 GB (91.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable

  Retrain complete

Training AutoGluon — Horizon: 1m
  Features: 125
  Train rows: 42,560  |  Valid rows: 8,540


Running DyStack sub-fit ...
Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/kaggle/working/ag_v2_1m/ds_sub_fit/sub_fit_ho"
Train Data Rows:    37831
Train Data Columns: 125
Label Column:       units_sold
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    26561.65 MB
	Train Data (Original)  Memory Usage: 35.36 MB (0.1% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 46 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting 

  [Val MAE 1m]: 10.9203
            model  score_test  score_val
CatBoost_2_BAG_L1  -10.303260  -7.980454
   XGBoost_BAG_L1  -10.458049  -7.676453
  CatBoost_BAG_L1  -10.547172  -7.989938
CatBoost_2_BAG_L2  -10.610807  -7.478051
LightGBM_2_BAG_L2  -10.727794  -7.579088
  Retraining on full data...


## Inference + Submission

In [ ]:
def parse_sample(sample):
    out = sample.copy()
    parsed = out["id"].str.extract(r"^(\d+)_(.+)_(\d{4}-\d{2}-\d{2})_(1d|7d|1m)$")
    out[["store_id","category","forecast_date","horizon"]] = parsed
    out["store_id"]       = out["store_id"].astype(int)
    out["forecast_date"]  = pd.to_datetime(out["forecast_date"])
    out["h"]              = out["horizon"].map(HORIZON_MAP)
    out["decision_date"]  = out["forecast_date"] - pd.to_timedelta(out["h"], unit="D")
    out["anchor_date"]    = out["decision_date"].clip(upper=pd.Timestamp("2024-10-31"))
    return out

test_meta = parse_sample(sample)
submission = sample[["id"]].copy()
submission["units_sold_predicted"] = np.nan

for h_str in ["1d","7d","1m"]:
    H = HORIZON_MAP[h_str]
    predictor_full, feat_cols = MODELS[h_str]
    panel = panels[h_str]

    sub_h = test_meta[test_meta["horizon"] == h_str].copy()

    # Merge panel features at anchor_date
    test_rows = sub_h.merge(
        panel.rename(columns={"date":"anchor_date"})[["store_id","category","anchor_date"] + feat_cols],
        on=["store_id","category","anchor_date"],
        how="left"
    ).fillna(0)

    # Override calendar features to reflect FORECAST date (future-known)
    fc_cal = date_all[cal_cols].rename(columns={"date":"forecast_date"})
    test_rows = test_rows.drop(columns=[c for c in cal_cols if c != "date" and c in test_rows.columns], errors="ignore")
    test_rows = test_rows.merge(fc_cal, on="forecast_date", how="left")

    preds = predictor_full.predict(test_rows[feat_cols]).clip(lower=0)
    submission.loc[test_meta["horizon"]==h_str, "units_sold_predicted"] = preds.values

submission["units_sold_predicted"] = submission["units_sold_predicted"].fillna(0).clip(lower=0)
submission.to_csv("submission_autogluon_v2.csv", index=False)
print(f"Saved submission_autogluon_v2.csv  shape={submission.shape}")
print(submission.describe())